# HF TDOA Analysis - Figures 11

This notebook uses the `hf_tdoa` library to analyze HF TDOA measurements.

In [1]:
import os
import datetime
import hf_tdoa as tdoa

%matplotlib inline

# Setup plotting style
tdoa.setup_plotting_style()

## Load WAV Files & Find Chirps

In [2]:
base_dir   = '../data'
data_set   = 'TX_WA5FRF_EL09nn-RX_N5DUP_EM02ch-40m'
sweep_rate = 10  # Hz/ms

# Path to chirp template - used for finding chirp locations via cross-correlation
template   = os.path.join('../data/templates', 'N6RFM_10Hz_per_ms_template.wav')

data_dir   = os.path.join(base_dir, data_set)
wavlist    = tdoa.obtain_wav_list(data_dir)

In [3]:
# Correlate each WAV with a known template chirp to identify chirp locations in each WAV file.
chirps = tdoa.find_chirps(wavlist, template, sweep_rate=sweep_rate, plot_correlation=False)


Finding Chirps via Cross-Correlation
  Path Info:      PathInfo(TX: WA5FRF (EL09nn), RX: N5DUP (EM02ch), Range: 318.0 km, Band: 7 MHz)
  Template:       N6RFM_10Hz_per_ms_template.wav
  Sweep Rate:     10 Hz/ms
  Files to Process: 32
  Chirps per File: Top 10



Finding chirps: 100%|██████████| 32/32 [00:01<00:00, 16.21file/s, Chirps=10, Max Corr=4.84e+02]

✓ Completed chirp detection: 32 files processed, 320 total chirps found



## Find TDOAs

In [4]:
debug_TDOAs = False

# Find TDOAs for the default propagation mode using the simplified API
# The mode configurations (filter limits, search limits, model coefficients, and plotting params)
# are now stored in hf_tdoa_lib.MODE_CONFIGS

# Process the default mode (2F2-1F2)
chirps = tdoa.find_TDOAs(chirps, mode_string='2F2-1F2',
                        plot_fft=debug_TDOAs, only_one=debug_TDOAs)

# Build the TDOA configuration dictionary with model coefficients and plotting parameters
# Only include the 2F2-1F2 mode for this analysis
# This will automatically print path information and calculated model coefficients
tdoa_dct = tdoa.build_tdoa_config(chirps, mode_strings=['2F2-1F2'])


Processing Mode: 2F2-1F2
  Filter Limits:  10.0 - 50.0 Hz
  Search Window:  -0.10 to 0.10 s offset
  Freq Range:     11.0 - 20.0 Hz
  Set Name:       2F2-1F2
  Sweep Rate:     10 Hz/ms
  Files to Process: 32



Finding TDOAs (2F2-1F2): 100%|██████████| 32/32 [00:07<00:00,  4.37file/s, Mean TDOA=1.75 ms]

✓ Completed processing 2F2-1F2: 32 files processed


Path Information
  PathInfo(TX: WA5FRF (EL09nn), RX: N5DUP (EM02ch), Range: 318.0 km, Band: 7 MHz)

TX Location: 29.563°N, -98.875°E (EL09nn)
RX Location: 32.313°N, -99.792°E (EM02ch)

Path Midpoint: 30.938°N, -99.327°E

Path Azimuth (TX→RX): -15.7°

Calculated TDOA Model Coefficients:
2F2-1F2      (2F2-1F2 ): slope= 143.0, intercept=  33.0



## Create Figure 11: Layer Heights with Eclipse Overlay

In [ ]:
# Access the path_info object for solar calculations
path_info = chirps.attrs['path_info']
solar_lat, solar_lon = path_info.get_midpoint()

# Set time limits to focus on the eclipse period
xlim = (datetime.datetime(2024, 4, 8, 14, 0), datetime.datetime(2024, 4, 8, 20, 30))

# Create enhanced plot with eclipse obscuration and TDOA CSV overlays
# Compare two verified manual analysis methods:
# - Manual Beatnote Analysis (blue dotted)
# - Manual Autocorrelation Analysis (orange dashed)
tdoa.plot_hmf2(chirps, tdoa_dct,
               xlim=xlim,
               solar_lat=solar_lat,
               solar_lon=solar_lon,
               overlay_eclipse=True,
               ionosonde_dct={'overlay_hmE': False},
               tdoa_csv_dct={
                   'csv_path_beatnote': '../data/CSVs/2024-04-08_TX_WA5FRF_EL09nn-RX_N5DUP_EM02ch-40m_manual_beatnote_analysis.csv',
                   'csv_path_autocorr': '../data/CSVs/2024-04-08_TX_WA5FRF_EL09nn-RX_N5DUP_EM02ch-40m_manual_autocorrelation_analysis.csv'
               },
               savefig='fig_11.jpg')